# Market Selection Audit

This notebook walks through the current `market_selection` logic step by step using the raw `market_universe` table.
Each filtering stage shows both counts and a sample of the rows that get dropped.

## Paper Framing

For the NeurIPS version, the goal of filtering is to exclude market types that instantiate a world we do not want the model to treat as part of the core real-world belief system.

The most defensible exclusion factors are:

- **Sports and esports markets.** These markets are driven by highly idiosyncratic game-level dynamics and are only weakly coupled to the broader real-world state we want a foundation-style model to represent.
- **Financial price-derived or benchmark-derived markets.** These markets resolve as near-direct functions of ticker prices, benchmark levels, or structured price thresholds rather than broader world events.
- **Attention, speech, and mention-count markets.** These markets are often mediated by platform-specific measurement rules and noisy textual resolution criteria rather than substantive external state.
- **Ultra-short recurring template markets.** These markets are repeated mechanical contracts and contribute little meaningful real-world variation.

Conversely, many popularity, consumption, polling, and macro-indicator markets remain plausibly world-grounded even if they are somewhat noisy. For that reason, this notebook treats `event_series_slug`, `event_slug`, and text patterns as an operationalization of a small set of semantic exclusion criteria, not as the conceptual definition of the protocol.

Import only the helpers we need. `polymarket_export` should already be installed in editable mode via `pip install -e ./polymarket_export`.

In [1]:
from pathlib import Path
import sqlite3

from IPython.display import display
import pandas as pd
import numpy as np

import polymarket_registry
from configs.resolved_dataset_domain_config import DOMAIN_PRIORITY

pd.set_option("display.max_colwidth", 160)
pd.set_option("display.max_rows", 100)
pd.set_option("display.max_columns", 100)

EXPORT_ROOT = Path(polymarket_registry.__file__).resolve().parents[1]
REPO_ROOT = EXPORT_ROOT.parent

EXPORT_ROOT, REPO_ROOT

(PosixPath('/Users/sneddy/research/polymarket_research/polymarket_export'),
 PosixPath('/Users/sneddy/research/polymarket_research'))

Configure the local database path and the main selection cutoff.

In [2]:
DB_PATH = (REPO_ROOT / "db" / "resolved_probability_dataset.sqlite").resolve()
MIN_CREATED_AT = "2025-01-01T00:00:00Z"
MIN_RESOLVED_VOLUME = 20_000.0
ENRICH_WITH_GAMMA_TAGS = True

DB_PATH


PosixPath('/Users/sneddy/research/polymarket_research/db/resolved_probability_dataset.sqlite')

Load `market_universe` dynamically so the notebook stays compatible with schema changes.
We keep a preferred column order for readability, but automatically include any newly added columns.



In [3]:
preferred_column_order = [
    "market_id",
    # "condition_id",
    "market_slug",
    "event_id",
    "event_slug",
    "event_title",
    "event_series_slug",
    "event_description",
    # "event_start_time",
    "event_score",
    "event_period",
    "event_series_id",
    "event_recurrence",
    "event_series_type",
    "question",
    "description",
    # "resolution_source",
    # "created_at",
    # "end_date",
    "volume_num",
    # "liquidity_num",
    "outcomes",
    "outcome_prices",
    # "clob_token_ids",
    # "closed_time",
    "uma_resolution_status",
    "neg_risk",
    "neg_risk_market_id",
    "group_item_title",
    # "synced_at_utc",
]

query = """
SELECT *
FROM market_universe
WHERE created_at IS NOT NULL
  AND created_at >= ?
ORDER BY volume_num DESC, created_at DESC
"""

with sqlite3.connect(DB_PATH) as conn:
    universe_columns = [row[1] for row in conn.execute("PRAGMA table_info(market_universe)").fetchall()]
    universe_df = pd.read_sql_query(query, conn, params=(MIN_CREATED_AT,))

extra_columns = [column for column in universe_df.columns if column not in preferred_column_order]
missing_preferred_columns = [column for column in preferred_column_order if column not in universe_df.columns]
ordered_columns = [column for column in preferred_column_order if column in universe_df.columns] + sorted(extra_columns)
universe_df = universe_df.loc[:, ordered_columns]

print({
    "db_path": str(DB_PATH),
    "universe_rows": len(universe_df),
    "min_created_at": MIN_CREATED_AT,
    "columns_in_table": len(universe_columns),
    "missing_preferred_columns": missing_preferred_columns,
    "extra_columns_detected": extra_columns,
})

universe_df.head(5)

{'db_path': '/Users/sneddy/research/polymarket_research/db/resolved_probability_dataset.sqlite', 'universe_rows': 732998, 'min_created_at': '2025-01-01T00:00:00Z', 'columns_in_table': 32, 'missing_preferred_columns': [], 'extra_columns_detected': ['condition_id', 'event_start_time', 'resolution_source', 'created_at', 'end_date', 'closed', 'archived', 'liquidity_num', 'clob_token_ids', 'closed_time', 'synced_at_utc']}


,market_id,market_slug,event_id,event_slug,event_title,event_series_slug,event_description,event_score,event_period,event_series_id,event_recurrence,event_series_type,question,description,volume_num,outcomes,outcome_prices,uma_resolution_status,neg_risk,neg_risk_market_id,group_item_title,archived,clob_token_ids,closed,closed_time,condition_id,created_at,end_date,event_start_time,liquidity_num,resolution_source,synced_at_utc
0,1640919,us-forces-enter-iran-by-april-30-899,158299,us-forces-enter-iran-by,US forces enter Iran by..?,NaN,"This market will resolve to “Yes” if active US military personnel physically enter Iran at any point by the listed date (ET). Otherwise, this market will re...",NaN,NaN,NaN,NaN,NaN,US forces enter Iran by April 30?,"This market will resolve to “Yes” if active US military personnel physically enter Iran at any point by the listed date (ET). Otherwise, this market will re...",2.690491e+08,"[""Yes"", ""No""]","[""1"", ""0""]",resolved,0.0,NaN,April 30,0,"[""2916184120206223749839849644877707470354946028257066951797428049170871002238"", ""76533108781962275310651165149634079251899733930834190485860627580128626747...",1,2026-04-09 00:28:21+00,0x6d0e09d0f04572d9b1adad84703458b0297bc5603b69dccbde93147ee4443246,2026-03-18T16:29:07.71272Z,2026-04-30T00:00:00Z,NaN,NaN,,2026-04-09T07:31:52Z
1,546814,will-zelenskyy-wear-a-suit-before-july,25044,will-zelenskyy-wear-a-suit-before-july,Will Zelenskyy wear a suit before July?,NaN,"This market will resolve to ""Yes"" if Volodymyr Zelenskyy is is photographed or videotaped wearing a suit between May 22 and June 30, 2025 ET. Otherwise, thi...",NaN,NaN,NaN,NaN,NaN,Will Zelenskyy wear a suit before July?,"This market will resolve to ""Yes"" if Volodymyr Zelenskyy is is photographed or videotaped wearing a suit between May 22 and June 30, 2025 ET. Otherwise, thi...",2.422312e+08,"[""Yes"", ""No""]","[""0"", ""1""]",resolved,0.0,NaN,,0,"[""103864131794756285503734468197278890131080300305704085735435172616220564121629"", ""343795817898955285602812182397592802372773053729787943248227774388244101...",1,2025-07-09 00:30:39+00,0x655e5ca101c466b6293aa15e06173b78b293221803d56e35551f708cd82eb352,2025-05-22T22:19:20.138837Z,2025-06-30T12:00:00Z,NaN,NaN,,2026-04-09T07:31:52Z
2,601697,fed-decreases-interest-rates-by-50-bps-after-january-2026-meeting,45883,fed-decision-in-january,Fed decision in January?,fed-interest-rates,The FED interest rates are defined in this market by the upper bound of the target federal funds range. The decisions on the target federal fund range are m...,NaN,NaN,35,monthly,single,Fed decreases interest rates by 50+ bps after January 2026 meeting?,The FED interest rates are defined in this market by the upper bound of the target federal funds range. The decisions on the target federal fund range are m...,2.350652e+08,"[""Yes"", ""No""]","[""0"", ""1""]",resolved,1.0,0x0d97e25f1830a3f18d5b4d43e6e1648d3b8b781e3c254a0b9d2c07a62f505100,50+ bps decrease,0,"[""11862165566757345985240476164489718219056735011698825377388402888080786399275"", ""7147885279027909544718299604907104079201075961766896979904917922910480057...",1,2026-01-28 22:53:04+00,0x17815081230e3b9c78b098162c33b1ffa68c4ec29c123d3d14989599e0c2e113,2025-09-17T16:22:35.837625Z,2026-01-28T00:00:00Z,NaN,NaN,,2026-04-09T07:31:52Z
3,601700,fed-increases-interest-rates-by-25-bps-after-january-2026-meeting,45883,fed-decision-in-january,Fed decision in January?,fed-interest-rates,The FED interest rates are defined in this market by the upper bound of the target federal funds range. The decisions on the target federal fund range are m...,NaN,NaN,35,monthly,single,Fed increases interest rates by 25+ bps after January 2026 meeting?,The FED interest rates are defined in this market by the upper bound of the target federal funds range. The decisions on the target federal fund range are m...,2.164557e+08,"[""Yes"", ""No""]","[""0"", ""1""]",resolved,1.0,0x0d97e25f1830a3f18d5b4d43e6e1648d3b8b781e3c254a0b9d2c07a62f505100

## Block 1. Ultra-Short Recurring Template Markets

In [4]:
universe_df.event_recurrence.value_counts()

event_recurrence
daily      417700
5m         104686
15m         71151
hourly      37888
weekly      27457
monthly      6596
annual       1200
Name: count, dtype: int64

In [5]:
universe_df[universe_df.event_recurrence == '5m'].event_series_slug.value_counts()

event_series_slug
btc-up-or-down-5m    27641
sol-up-or-down-5m    25684
eth-up-or-down-5m    25681
xrp-up-or-down-5m    25680
Name: count, dtype: int64

In [6]:
universe_df[universe_df.event_recurrence == '15m'].event_series_slug.value_counts()

event_series_slug
btc-up-or-down-15m    19929
eth-up-or-down-15m    19922
xrp-up-or-down-15m    15658
sol-up-or-down-15m    15642
Name: count, dtype: int64

In [7]:
universe_df[universe_df.event_recurrence == 'hourly'].event_series_slug.value_counts()

event_series_slug
eth-up-or-down-hourly            7207
btc-up-or-down-hourly            7201
solana-up-or-down-hourly         7082
xrp-up-or-down-hourly            6898
bitcoin-multi-strikes-hourly     4569
ethereum-multi-strikes-hourly    4553
bitcoin-hourly-up-or-down         329
bitcoin-up-or-down-hourly          25
eth-hourly-up-or-down              24
Name: count, dtype: int64

In [8]:
universe_df[universe_df.event_recurrence == 'daily'].event_series_slug.value_counts()

event_series_slug
nba-2026                                      45145
ncaa-cbb                                      31783
counter-strike                                29129
dota-2                                        28497
atp                                           25071
                                              ...  
hurricane-form                                    1
chuck-schumer-meet-trump                          1
republican-house-odds-up-or-down-this-week        1
trump-invoke-war-powers                           1
btc-up-or-down-hourly                             1
Name: count, Length: 435, dtype: int64

In [ ]:
universe_df["category"] = None

# recurring short-horizon templates
universe_df.loc[
    universe_df["event_recurrence"].isin(["5m", "15m", "hourly"]),
    "category",
] = "short_recurrence"

# up/down markets are another mechanical recurring template
mask_updown = universe_df["outcomes"].astype(str).str.strip().eq('["Up", "Down"]')
universe_df.loc[
    mask_updown & universe_df["category"].isna(),
    "category",
] = "short_recurrence"

universe_df["category"].fillna("unknown").value_counts()


category
unknown             477435
short_recurrence    255563
Name: count, dtype: int64

: 

## Block 2. Financial Price-Derived Or Benchmark-Derived Markets

In [ ]:
series_slug = universe_df["event_series_slug"].fillna("").astype(str).str.lower().str.strip()

# structured price constructions
slug_price_structures = series_slug.str.contains(
    r"(multi-strikes|neg-risk|hit-price|weekly-brackets|monthly-prices)",
    regex=True,
    na=False,
)

universe_df.loc[
    slug_price_structures & universe_df["category"].isna(),
    "category",
] = "quant_price_structures"

# market-target / benchmark-like metrics
slug_target_metrics = series_slug.str.contains(
    r"(?:eth-weeklies|btc-weeklies|sol-weeklies|doge-monthly|mstr-weeklies|"
    r"ethereum-etf-flows-daily|bitcoin-etf-flows-daily|"
    r"crude-oil-cl-hit|will-silver-si-hit|what-will-gold-gc-hit|"
    r"mag-7-weekly|largest-company|second-largest-company|ipo-closing-market-cap)",
    regex=True,
    na=False,
)

universe_df.loc[
    slug_target_metrics & universe_df["category"].isna(),
    "category",
] = "quant_price_structures"

universe_df["category"].fillna("unknown").value_counts()


In [ ]:
universe_df[universe_df.category == 'quant_price_structures'].event_series_slug.value_counts()

## Block 3. Attention, Speech, And Mention-Count Markets

In [ ]:
series_slug = universe_df["event_series_slug"].fillna("").astype(str).str.lower().str.strip()

attention_mask = series_slug.str.contains(
    r"(tweets?|mentions?|truth-social|pmqs|mrbeast-views|trump-truths|trump-post-weekly|trump-talk-monthly|all-in-podcast|rogan-mentions|andrew-tate-tweets)",
    regex=True,
    na=False,
)

universe_df.loc[
    attention_mask & universe_df["category"].isna(),
    "category",
] = "attention_social_metrics"

universe_df.category.value_counts()


## Block 4. Weather Markets

In [ ]:
weather_mask = universe_df.event_series_slug.apply(lambda x: "weather" in str(x).lower())
universe_df.loc[weather_mask, "category"] = "weather"
universe_df['category'].fillna('unknown').value_counts()

## Block 5. Sports And Esports Markets

In [ ]:
import ast
import re


def _s(df: pd.DataFrame, col: str) -> pd.Series:
    if col not in df.columns:
        return pd.Series("", index=df.index)
    return df[col].fillna("").astype(str).str.lower().str.strip()


def _parse_outcomes(v):
    if isinstance(v, list):
        return [str(x).strip().lower() for x in v]
    if isinstance(v, str):
        t = v.strip()
        if not t:
            return None
        try:
            x = ast.literal_eval(t)
            if isinstance(x, list):
                return [str(i).strip().lower() for i in x]
        except Exception:
            return None
    return None


def _has_slug_token(series_slug: pd.Series, token: str) -> pd.Series:
    t = str(token).strip().lower()
    return series_slug.str.contains(rf"(?:^|-){re.escape(t)}(?:-|$)", regex=True, na=False)


def _slug_signals(df: pd.DataFrame) -> tuple[pd.Series, pd.Series, pd.Series]:
    series_slug = _s(df, "event_series_slug")
    event_slug = _s(df, "event_slug")
    primary_slug = series_slug.where(series_slug.ne(""), event_slug)
    slug_blob = (series_slug + " " + event_slug).str.strip()
    return primary_slug, event_slug, slug_blob


CYBER_RE = (
    r"(?:counter-strike|cs2|csgo|dota|league-of-legends|valorant|honor-of-kings|overwatch|"
    r"starcraft-2|mobile-legends-bang-bang|call-of-duty|esports|lck|lpl)"
)

SPORT_RE = (
    r"(?:nba|nfl|mlb|nhl|ncaa|cbb|cwbb|cfb|atp|wta|efl-championship|fa-cup|ucl-|uel-|efl-|"
    r"europa-conference-league|coupe-de-france|khl|ahl|shl|dehl|cehl|"
    r"japan-j2-league|japan-j-league|serie-b|ligue-2|primeira-liga|saudi-professional-league|"
    r"fifa-friendly|primera-a|primera-division|primera-divisin-argentina|"
    r"scottish-premiership|ukraine-premier-liha|womens-t20-world-cup-qualifier|"
    r"liga-1|egypt-1|romania-1|czechia-1|"
    r"(?:mls|la-liga|ligue-1|bundesliga|premier-league|serie-a)(?:-\d{4})?|"
    r"(?:mex|tur|ere)-\d{4}|cricket|soccer|basketball)"
)


def detect_cybersport(df: pd.DataFrame) -> pd.Index:
    primary_slug, event_slug, slug_blob = _slug_signals(df)
    text_blob = _s(df, "market_slug") + " " + _s(df, "event_title") + " " + _s(df, "question")
    event_score = _s(df, "event_score")
    event_period = _s(df, "event_period")

    by_slug = (
        primary_slug.str.contains(CYBER_RE, regex=True, na=False)
        | event_slug.str.contains(CYBER_RE, regex=True, na=False)
        | slug_blob.str.contains(CYBER_RE, regex=True, na=False)
    )
    by_text = text_blob.str.contains(CYBER_RE, regex=True, na=False)
    by_bo = event_score.str.contains(r"\bbo\d+\b", regex=True, na=False) | event_period.str.contains(
        r"\bbo\d+\b", regex=True, na=False
    )

    mask = by_slug | by_text | by_bo
    return df.loc[mask, "market_id"]


def detect_sport(df: pd.DataFrame) -> pd.Index:
    primary_slug, event_slug, slug_blob = _slug_signals(df)
    text_blob = _s(df, "market_slug") + " " + _s(df, "event_title") + " " + _s(df, "question")
    event_score = _s(df, "event_score")

    by_slug = (
        primary_slug.str.contains(SPORT_RE, regex=True, na=False)
        | event_slug.str.contains(SPORT_RE, regex=True, na=False)
        | slug_blob.str.contains(SPORT_RE, regex=True, na=False)
    )
    # минимально добавляем точные slug-токены, чтобы не ловить venezuela/nuclear
    by_slug_token = (
        _has_slug_token(primary_slug, "uel")
        | _has_slug_token(primary_slug, "ucl")
        | _has_slug_token(primary_slug, "efl")
        | _has_slug_token(event_slug, "uel")
        | _has_slug_token(event_slug, "ucl")
        | _has_slug_token(event_slug, "efl")
    )

    by_text = text_blob.str.contains(SPORT_RE, regex=True, na=False)
    by_score = event_score.ne("")

    outcomes_norm = (
        df["outcomes"].map(_parse_outcomes)
        if "outcomes" in df.columns
        else pd.Series([None] * len(df), index=df.index)
    )
    by_outcomes = outcomes_norm.map(
        lambda x: isinstance(x, list) and x in (["over", "under"], ["odd", "even"], ["favorite", "underdog"])
    )

    mask = by_slug | by_slug_token | by_text | by_score | by_outcomes
    return df.loc[mask, "market_id"]


In [ ]:
missing_mask = universe_df["category"].isna() | universe_df["category"].astype(str).str.strip().eq("")

cyber_ids = set(detect_cybersport(universe_df))
sport_ids = set(detect_sport(universe_df))

# приоритет: cybersport -> sport
universe_df.loc[missing_mask & universe_df["market_id"].isin(cyber_ids), "category"] = "cybersport"

missing_mask = universe_df["category"].isna() | universe_df["category"].astype(str).str.strip().eq("")
universe_df.loc[missing_mask & universe_df["market_id"].isin(sport_ids), "category"] = "sport"

universe_df["category"].fillna("unknown").value_counts()


## Final Volume Screen (> 20,000 USD)

In [ ]:
rest_pre_volume = universe_df[universe_df.category.isnull()].copy()
rest = rest_pre_volume[rest_pre_volume["volume_num"] > MIN_RESOLVED_VOLUME].copy()

print({
    "pre_volume_rest_rows": len(rest_pre_volume),
    "post_volume_rest_rows": len(rest),
    "dropped_by_volume": len(rest_pre_volume) - len(rest),
    "min_volume_usd": MIN_RESOLVED_VOLUME,
})


## Residual Review After All Filters

In [ ]:
vc = rest.event_series_slug.value_counts()
vc.head(100)


In [ ]:
rest.sample(5)


In [ ]:
rest.event_title.str.lower().sample(30)


In [ ]:
rest_pre_volume[rest_pre_volume.volume_num <= MIN_RESOLVED_VOLUME].sort_values("volume_num", ascending=False).head(20)


In [ ]:
(universe_df["volume_num"] <= MIN_RESOLVED_VOLUME).mean(), (rest_pre_volume["volume_num"] <= MIN_RESOLVED_VOLUME).mean()


In [ ]:
rest[rest.event_series_slug.isnull()].sample(10)
